# SparkRules — Getting Started

This notebook shows how to define rules in DRL, evaluate facts, and inspect results.

```bash
pip install sparkrules
```

In [ ]:
from sparkrules.executor import RuleExecutor
from sparkrules.parser import parse

## 1. Define a simple rule

Rules use Drools-style DRL syntax: `when` matches conditions, `then` sets outputs.

In [ ]:
drl = """
rule "high-value-order"
  salience 10
  reason_codes ["HV001"]
  when
    $order : Order( amount > 1000 )
  then
    result.risk = "high";
    result.review_required = true;
end
"""

# Parse and inspect the AST
ast = parse(drl)
print(f"Rule name: {ast.name}")
print(f"Salience: {ast.salience}")
print(f"Reason codes: {ast.reason_codes}")
print(f"Conditions: {len(ast.when)} pattern(s)")
print(f"Actions: {len(ast.then)} action(s)")

## 2. Evaluate a fact

In [ ]:
executor = RuleExecutor()

# Fact that matches the rule
result = executor.run({"amount": 1500, "region": "US"}, drl)

print(f"Fired: {result.fired}")
print(f"Action output: {result.action_output}")
print(f"Bound fields: {result.bound_fields}")
print(f"Reason codes: {result.reason_codes}")

In [ ]:
# Fact that does NOT match
result2 = executor.run({"amount": 500, "region": "US"}, drl)

print(f"Fired: {result2.fired}")
print(f"Action output: {result2.action_output}")

## 3. Multiple rules with salience ordering

In [ ]:
from sparkrules.parser import parse_rules
from sparkrules.runtime.rule_chain import run_rule_chain

multi_drl = """
rule "premium-customer"
  salience 20
  when
    $c : Customer( tier == "gold" )
  then
    result.discount = 15;
end

rule "standard-discount"
  salience 10
  when
    $c : Customer( tier == "silver" )
  then
    result.discount = 5;
end
"""

chain_result = run_rule_chain(
    [multi_drl],
    {"tier": "gold", "years": 3},
)

for step in chain_result:
    print(f"Rule: {step['rule_name']}, Fired: {step['fired']}, Output: {step.get('action_output', {})}")

## 4. Explainable results

Every evaluation returns structured data — bound fields, action outputs, reason codes — making rules auditable and debuggable.

In [ ]:
import json

# Pretty-print the full result
print(json.dumps({
    "fact_id": result.fact_id,
    "rule_id": result.rule_id,
    "fired": result.fired,
    "reason_codes": list(result.reason_codes),
    "bound_fields": result.bound_fields,
    "action_output": result.action_output,
    "error": result.error_message,
}, indent=2))